# Unsloth Fine-Tuning on CUDA

Fine-tune a Llama 3.2 model using Unsloth + LoRA on a CUDA GPU.

- Model loaded directly from Hugging Face via Unsloth's `FastLanguageModel`
- Dataset fetched directly from Hugging Face Hub
- All metrics logged to **TensorBoard**
- Requires CUDA GPU (Unsloth does not support CPU/MPS)


In [ ]:
# Install required packages (CUDA environment)
%pip install -q unsloth datasets trl tensorboard huggingface_hub seaborn


# Step 1: Load Model

Load `unsloth/Llama-3.2-3B` directly from Hugging Face using Unsloth's `FastLanguageModel`.
Unsloth automatically patches the model for 2x faster training with reduced VRAM usage.


In [ ]:
from unsloth import FastLanguageModel

HF_MODEL_ID = "unsloth/Llama-3.2-3B"
max_seq_length = 2048

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_MODEL_ID,
    max_seq_length=max_seq_length,
    dtype=None,          # auto-detect bfloat16 / float16
    load_in_4bit=False,   # 4-bit quantisation for lower VRAM
)

print(f"✓ Loaded: {HF_MODEL_ID}")
print(f"✓ Max sequence length: {max_seq_length}")


# Step 2: Attach LoRA Adapters

Attach LoRA adapters using Unsloth's optimised `get_peft_model`. Only the adapter weights are trained, keeping VRAM usage low.

Parameters: rank=16, alpha=16.


In [ ]:
ft_model = FastLanguageModel.get_peft_model(
    base_model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves extra VRAM
    random_state=3407,
)

total_params     = sum(p.numel() for p in ft_model.parameters())
trainable_params = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
trainable_pct    = trainable_params / total_params * 100

print(f"✓ LoRA adapters attached")
print(f"✓ Total parameters:     {total_params:,}")
print(f"✓ Trainable parameters: {trainable_params:,}  ({trainable_pct:.2f}% of total)")
print(f"✓ Frozen parameters:    {total_params - trainable_params:,}  ({100 - trainable_pct:.2f}% of total)")


# Step 3: Load Dataset

Load the `ServiceNow-AI/R1-Distill-SFT` dataset directly from Hugging Face Hub.

A **character-length proxy** is used to cheaply pre-filter obviously-too-long examples without tokenizing the full dataset. A `POOL_MULTIPLIER` oversamples beyond `subset_size` to account for examples that will be dropped later by the precise token filter in Step 4a (EDA).

Precise token-length filtering happens in Step 4a after tokenization is already computed for EDA — no redundant work.


In [ ]:
from datasets import load_dataset

HF_DATASET_ID = "ServiceNow-AI/R1-Distill-SFT"
subset_size = 20480    # final number of examples after precise filtering in EDA
POOL_MULTIPLIER = 1.5  # oversample to account for examples dropped by token filter
CHARS_PER_TOKEN = 4    # conservative proxy: ~4 chars per token for this dataset
char_limit = max_seq_length * CHARS_PER_TOKEN

dataset = load_dataset(HF_DATASET_ID, "v0", split="train")
print(f"✓ Full dataset: {len(dataset):,} examples")

# ── Cheap char-length pre-filter (no tokenization) ───────────────────────────
# Drops examples whose response alone already exceeds the char proxy limit.
# This is O(n) string-length check — very fast.
dataset = dataset.filter(
    lambda row: len(row["reannotated_assistant_content"]) < char_limit,
    num_proc=4,
)
print(f"✓ After char-length pre-filter (<{char_limit:,} chars): {len(dataset):,} examples")

# ── Oversample pool then trim — precise filter happens later in EDA ───────────
pool_size = min(int(subset_size * POOL_MULTIPLIER), len(dataset))
dataset = dataset.select(range(pool_size))

print(f"✓ Pool selected: {pool_size:,} examples (will trim to {subset_size:,} after token filter in Step 4a)")
print(f"✓ Columns: {dataset.column_names}")

# Step 4: Format Dataset

Apply the Llama 3.1 chat template to each example.
Each row is converted into a user/assistant conversation and tokenised to match inference-time format.


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

def formatting_prompts_func(examples):
    texts = []
    for problem, response in zip(examples["problem"], examples["reannotated_assistant_content"]):
        convo = [
            {"role": "user", "content": problem},
            {"role": "assistant", "content": response},
        ]
        texts.append(tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

print("✓ Dataset formatted with chat template")

# Step 4a: Exploratory Data Analysis

Quick inspection of the formatted dataset before training:

1. **Sample comparison** — raw vs formatted text for the first example
2. **Context length distribution** — token count of the full formatted prompt
3. **Response length distribution** — token count of the assistant reply only


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sns.set_theme(style="whitegrid", context="talk", palette="muted")

# ── 1. Sample: raw vs formatted ──────────────────────────────────────────────
sample = dataset[0]
print("=" * 70)
print("RAW PROBLEM:")
print(sample["problem"])
print("\nRAW RESPONSE:")
print(sample["reannotated_assistant_content"])
print("\n" + "=" * 70)
print("FORMATTED TEXT:")
print(sample["text"])
print("=" * 70)

# ── 2 & 3. Compute token lengths (tokenization already needed for EDA) ────────
print(f"\nComputing token lengths for {len(dataset):,} examples...")

context_lengths  = [len(tokenizer.encode(row["text"], add_special_tokens=False))
                    for row in dataset]
response_lengths = [len(tokenizer.encode(row["reannotated_assistant_content"], add_special_tokens=False))
                    for row in dataset]

df = pd.DataFrame({
    "context_length":  context_lengths,
    "response_length": response_lengths,
    "idx": range(len(dataset)),
})

print(f"\n📊 Before token filter ({len(df):,} examples):")
print(df[["context_length", "response_length"]].describe().round(1).to_string())

# ── Precise token-length filter (drop samples exceeding max_seq_length) ───────
valid_mask = df["context_length"] < max_seq_length
dropped = (~valid_mask).sum()
df_valid = df[valid_mask].reset_index(drop=True)
print(f"\n✂  Dropped {dropped:,} examples exceeding {max_seq_length} tokens  "
      f"({dropped / len(df) * 100:.1f}%)")

# Trim to final subset_size
if len(df_valid) > subset_size:
    df_valid = df_valid.iloc[:subset_size]

# Apply filtered indices back to the dataset
dataset = dataset.select(df_valid["idx"].tolist())
context_lengths  = df_valid["context_length"].tolist()
response_lengths = df_valid["response_length"].tolist()
df_plot = df_valid[["context_length", "response_length"]]

print(f"✓ Final dataset: {len(dataset):,} examples (target was {subset_size:,})")
print(f"\n📊 After token filter:")
print(df_plot.describe().round(1).to_string())

# ── 2. Context Length Distribution ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.histplot(data=df_plot, x="context_length", bins=60, kde=True,
             color="steelblue", ax=axes[0])
axes[0].axvline(max_seq_length, color="red", linestyle="--", linewidth=1.5,
                label=f"max_seq_length={max_seq_length}")
axes[0].set_title("Distribution of Context Lengths")
axes[0].set_xlabel("Context Length (tokens)")
axes[0].set_ylabel("Frequency")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[0].legend()

# ── 3. Response Length Distribution ──────────────────────────────────────────
sns.histplot(data=df_plot, x="response_length", bins=60, kde=True,
             color="coral", ax=axes[1])
axes[1].set_title("Distribution of Response Lengths")
axes[1].set_xlabel("Response Length (tokens)")
axes[1].set_ylabel("Frequency")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.suptitle("Token Length Distributions (post-filter)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


# Step 5: Configure Trainer

Configure TRL's `SFTTrainer`. Metrics are reported to **TensorBoard** (`./outputs/logs`).
Launch TensorBoard with: `tensorboard --logdir outputs/logs`

**Prompt masking** is applied via `train_on_responses_only`: labels for all user-prompt tokens are set to `-100` so the loss is computed **only on the assistant response**, not the instruction. This is the correct approach for instruction fine-tuning.


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only
import torch

MAX_STEPS_QUICK_RUN = -1          # set to a positive number for a quick smoke-test
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4  # effective batch size = 8
WARMUP_STEPS = 100
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 2e-4

trainer = SFTTrainer(
    model=ft_model,
    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="outputs",
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        max_steps=MAX_STEPS_QUICK_RUN,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="tensorboard",
        logging_dir="outputs/logs",
    ),
)

# ── Prompt masking: only compute loss on assistant response tokens ────────────
# Tokens belonging to the user instruction are set to label=-100 so the model
# does NOT learn to predict them — only the assistant reply contributes to loss.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

print("✓ Trainer configured with response-only loss (prompt tokens masked)")
print(f"✓ Reporting metrics to TensorBoard → ./outputs/logs")


# Step 6: Train the Model

VRAM stats are printed before training starts. All metrics (loss, learning rate, throughput) are written to TensorBoard in real time.


In [ ]:
import torch

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"Device: {gpu_stats.name}  |  Total VRAM: {max_memory} GB  |  Reserved before training: {start_gpu_memory} GB\n")

trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n✓ Training complete!")
print(f"Final loss:       {trainer_stats.training_loss:.4f}")
print(f"Peak VRAM:        {used_memory} GB ({round(used_memory / max_memory * 100, 1)}% of {max_memory} GB)")
print(f"Training delta:   {round(used_memory - start_gpu_memory, 3)} GB")
print(f"\nView metrics:  tensorboard --logdir runs")


# Step 7: Inference Sanity Check

Switch the model to inference mode and run a quick reasoning prompt to verify the fine-tuned weights.


In [ ]:
# ── Diagnostic: what does the tokenizer treat as end-of-turn? ────────────────

# 1. What are the canonical EOS/EOT token IDs?
print("eos_token     :", repr(tokenizer.eos_token),     "→ id", tokenizer.eos_token_id)
print("bos_token     :", repr(tokenizer.bos_token),     "→ id", tokenizer.bos_token_id)
eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
print("<|eot_id|>    :", "→ id", eot_id)

# 2. What do the last ~10 tokens of a formatted training example look like?
#    This tells us what token IDs the model actually saw at the end of every response.
sample_text = dataset[0]["text"]
token_ids = tokenizer.encode(sample_text)
tail_ids   = token_ids[-10:]
tail_toks  = tokenizer.convert_ids_to_tokens(tail_ids)
print("\nLast 10 tokens of dataset[0]['text']:")
for tid, tok in zip(tail_ids, tail_toks):
    print(f"  {tid:>7}  {repr(tok)}")

In [ ]:
from unsloth import FastLanguageModel
from transformers import TextStreamer

FastLanguageModel.for_inference(ft_model)

test_problem = (
    "If Alex is taller than Blake, Blake is taller than Casey, "
    "and Casey is taller than Dana, who is the shortest person?"
)

messages = [{"role": "user", "content": test_problem}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
_ = ft_model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=max_seq_length,      # generate up to max_seq_length tokens in the response
    use_cache=True,
    temperature=0.6,
    top_p=0.9,
    eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>"),  # 128009
)


# Step 8: Save Artifacts

Save the LoRA adapter weights locally. Optionally push to the Hugging Face Hub or export to GGUF.


In [ ]:
ft_model.save_pretrained("outputs/adapters/")
tokenizer.save_pretrained("outputs/adapters/")
print("✓ LoRA adapters saved to ./outputs/adapters/")

from huggingface_hub import notebook_login, hf_api
notebook_login()

# Optional: push to Hugging Face Hub
ft_model.push_to_hub("tangowhisky16/llama-3.2-3B-think-fp16-v2")
tokenizer.push_to_hub("tangowhisky16/llama-3.2-3B-think-fp16-v2")

# Optional: upload entire folder (adapters + training logs) to Hugging Face Hub
hf_api.upload_folder(
    folder_path="outputs",
    repo_id="tangowhisky16/llama-3.2-3B-think-fp16-v2",
    repo_type="model"
)

# ── Save full fused model (merged base + adapters) ───────────────────────────
# Merges LoRA weights into the base model and saves in float16.
# This produces a standalone model that can be loaded without PEFT/Unsloth.
ft_model.save_pretrained_merged(
    "fused_model",
    tokenizer,
    save_method="merged_16bit",   # options: "merged_16bit" | "merged_4bit" | "lora"
)
print("✓ Fused model saved to fused_model")

# Optional: push fused model to Hugging Face Hub
ft_model.push_to_hub_merged("tangowhisky16/llama-3.2-3B-think-fp16-v2-fused", tokenizer, save_method="merged_16bit")



# Optional: export to GGUF (requires llama.cpp)
# ft_model.save_pretrained_gguf("outputs/lora_model_gguf", tokenizer, quantization_method="q4_k_m")


# Step 9: Evaluation on GSM8K

Compare base model vs fine-tuned model on 100 GSM8K math reasoning samples.

- **Decoding:** greedy (temperature=0, deterministic)
- **Metric:** exact match on extracted final number
- **Same prompt** used for both models


In [ ]:
import re
import time
import warnings
import os
from tqdm.auto import tqdm
from datasets import load_dataset
from unsloth import FastLanguageModel

# ── Config ────────────────────────────────────────────────────────────────────
N_SAMPLES  = 100
LOG_PATH   = "outputs/log.txt"
EOT_ID     = tokenizer.convert_tokens_to_ids("<|eot_id|>")
GEN_CFG    = dict(max_new_tokens=max_seq_length,        # increase if answers get cut off
                  do_sample=False,
                  temperature=1.0,
                  top_p=1.0,
                  eos_token_id=EOT_ID,
                  use_cache=True)

PROMPT_TEMPLATE = "Solve the following math problem. Give only the final numeric answer on the last line.\n\nProblem: {question}\n\nAnswer:"

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)

# Suppress the spurious max_new_tokens / max_length conflict warning from HF
# (HF emits this via logging, not warnings.warn, so must use logging to suppress it)
import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

os.makedirs("outputs", exist_ok=True)

def log(msg, fh):
    print(msg)
    fh.write(msg + "\n")

# ── Verify base_model vs ft_model identity ────────────────────────────────────
print("=== Model identity check ===")
print(f"base_model is ft_model          : {base_model is ft_model}")
print(f"id(base_model)                  : {id(base_model)}")
print(f"id(ft_model)                    : {id(ft_model)}")
print(f"ft_model wraps base_model       : {getattr(ft_model, 'base_model', None) is not None}")
inner = getattr(getattr(ft_model, 'base_model', None), 'model', None)
print(f"ft_model.base_model.model is base_model: {inner is base_model}")
if hasattr(ft_model, 'peft_config'):
    print(f"Adapter configs                 : {list(ft_model.peft_config.keys())}")
print()

# ── Load GSM8K ────────────────────────────────────────────────────────────────
gsm = load_dataset("openai/gsm8k", "main", split="test").select(range(N_SAMPLES))

def get_ground_truth(answer_str):
    m = re.search(r"####\s*(-?\d+\.?\d*)", answer_str)
    return m.group(1) if m else None

def extract_prediction(text):
    # Strip <think>...</think> first so we pick the answer, not a number inside reasoning
    text = THINK_RE.sub("", text).strip()
    matches = re.findall(r"-?\d+\.?\d*", text)
    return matches[-1] if matches else None

def evaluate(model, label, fh, track_think=False, use_adapter=True):
    """
    use_adapter=False  → disables LoRA layers before inference (base model behaviour).
    use_adapter=True   → keeps LoRA layers active (fine-tuned behaviour).
    """
    if not use_adapter:
        model.disable_adapter_layers()
        print(f"  [adapter DISABLED for '{label}']")
    else:
        model.enable_adapter_layers()
        print(f"  [adapter ENABLED  for '{label}']")

    FastLanguageModel.for_inference(model)
    correct      = 0
    total_tokens = 0
    think_count  = 0
    hit_max      = 0

    for i, row in enumerate(tqdm(gsm, desc=label[:30], unit="sample")):
        prompt = PROMPT_TEMPLATE.format(question=row["question"])
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")

        with __import__("torch").no_grad():
            out_ids = model.generate(input_ids=inputs, **GEN_CFG)

        new_ids = out_ids[0][inputs.shape[-1]:]
        n_tok   = len(new_ids)
        total_tokens += n_tok
        if n_tok >= GEN_CFG["max_new_tokens"]:
            hit_max += 1

        output = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

        if track_think and THINK_RE.search(output):
            think_count += 1

        pred = extract_prediction(output)
        gt   = get_ground_truth(row["answer"])
        correct_flag = pred is not None and gt is not None and pred == gt
        if correct_flag:
            correct += 1

        # ── Per-sample printout (stdout + log file) ───────────────────────────
        verdict = "✓" if correct_flag else "✗"
        # print to console and log file with same formatting
        log(f"[{label} | sample {i+1}/{N_SAMPLES}] {verdict}  pred={pred}  gt={gt}  tokens={n_tok}", fh)
        
        # write question and output
        fh.write(f"Q: {row['question']}\n")
        fh.write(f"A: {output}\n")

    log(f"\n{'─'*70}", fh)

    acc     = correct / N_SAMPLES * 100
    avg_tok = total_tokens / N_SAMPLES
    print(f"\n  → hit max_new_tokens ({GEN_CFG['max_new_tokens']}): {hit_max}/{N_SAMPLES} samples")
    return dict(label=label, correct=correct, acc=acc,
                total_tokens=total_tokens, avg_tokens=avg_tok,
                think_count=think_count if track_think else None)

# ── Run evaluation (both via ft_model, toggling adapter on/off) ───────────────
with open(LOG_PATH, "w") as fh:
    results = [
        evaluate(ft_model, "Base model (adapter OFF)", fh, track_think=False, use_adapter=False),
        evaluate(ft_model, "Fine-tuned (adapter ON)",  fh, track_think=True,  use_adapter=True),
    ]

print(f"\n✓ Per-sample log saved to {LOG_PATH}")

# ── Report ────────────────────────────────────────────────────────────────────
W = 36
print(f"\n{'Model':{W}}  {'Correct':>8}  {'Accuracy':>9}  {'Total tok':>10}  {'Avg tok/resp':>13}")
print("-" * (W + 48))
for r in results:
    print(f"{r['label']:{W}}  {r['correct']:>5}/{N_SAMPLES}  {r['acc']:>8.1f}%"
          f"  {r['total_tokens']:>10,}  {r['avg_tokens']:>12.1f}")

print("-" * (W + 48))
base, ft = results
print(f"{'Accuracy improvement':{W}}  {ft['acc'] - base['acc']:>+18.1f}%")
print(f"{'Avg tokens delta':{W}}  {ft['avg_tokens'] - base['avg_tokens']:>+18.1f}")

tc = ft["think_count"]
print(f"\n<think> block present (fine-tuned):  {tc}/{N_SAMPLES}  ({tc / N_SAMPLES * 100:.1f}% of responses)")
print(f"\nDataset: GSM8K test ({N_SAMPLES} samples) | Metric: exact match | Decoding: greedy")
